In [1]:
import os,sys
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/V-Max_Hackerz/
sys.path.insert(0,'/content/drive/MyDrive/V-Max_Hackerz/notebook')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/V-Max_Hackerz


In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

PROJECT_ROOT = Path("..").resolve()

TRAIN_DIR = PROJECT_ROOT / "student_resource"/"dataset" / "train"
SRC_DIR = PROJECT_ROOT / "code" / "business_entity_resolution" / "src"

sys.path.insert(0, str(SRC_DIR))

from business_entity_resolution.preprocessing.normalization import (
    normalize_name,
    normalize_address
)

print(PROJECT_ROOT)

D:\V-Max_Hackerz


In [2]:
s1 = pd.read_csv(
    TRAIN_DIR / "train_source1.tsv",
    sep="\t",
    dtype=str
)
s2 = pd.read_csv(
    TRAIN_DIR / "train_source2.tsv",
    sep="\t",
    dtype=str
)

s3 = pd.read_csv(
    TRAIN_DIR / "train_source3.tsv",
    sep="\t",
    dtype=str
)

gt = pd.read_csv(
    TRAIN_DIR / "train_ground_truth.tsv",
    sep="\t",
    dtype=str
)

print("S1:", s1.shape)
print("S2:", s2.shape)
print("S3:", s3.shape)
print("GT:", gt.shape)

S1: (2206821, 4)
S2: (5034616, 4)
S3: (5285603, 4)
GT: (2206821, 2)


In [3]:
from sklearn.model_selection import train_test_split

train_s1_ids, val_s1_ids = train_test_split(
    s1["entity_id"],
    test_size=0.20,
    random_state=42
)

train_s1_ids = set(train_s1_ids)
val_s1_ids = set(val_s1_ids)

print("Training S1:", len(train_s1_ids))
print("Validation S1:", len(val_s1_ids))

Training S1: 1765456
Validation S1: 441365


In [4]:
candidate_source = pd.concat(
    [
        s2[["entity_id", "business_name", "business_address", "country"]],
        s3[["entity_id", "business_name", "business_address", "country"]],
    ],
    ignore_index=True
)

candidate_source = candidate_source.set_index("entity_id")

print(candidate_source.shape)

(10320219, 3)


In [ ]:
ground_truth = {}

for row in gt.itertuples(index=False):

    if pd.isna(row.matched_entity_ids) or not row.matched_entity_ids:
        ground_truth[row.source1_entity_id] = set()
    else:
        ground_truth[row.source1_entity_id] = set(
            row.matched_entity_ids.split(",")
        )

print("Ground-truth S1 entities:", len(ground_truth))
list(ground_truth.items())[:3]

In [ ]:
from collections import defaultdict, Counter
from itertools import combinations

name_index = defaultdict(list)

for row in candidate_source.reset_index().itertuples(index=False):

    name = normalize_name(row.business_name)

    if not name:
        continue

    name_index[
        (row.country, name)
    ].append(row.entity_id)

In [ ]:
address_index = defaultdict(list)

for row in candidate_source.reset_index().itertuples(index=False):

    address = normalize_address(row.business_address)

    if not address:
        continue

    address_index[
        (row.country, address)
    ].append(row.entity_id)

In [ ]:
NAME_STOPWORDS = {
    "private","limited","corporation","company","incorporated",
    "llc","llp","pllc","corp","inc","co",
    "group","holding","holdings","services","service",
    "solutions","enterprises","international","global",
    "partners","associates","consultancy","consultants",
    "and","india"
}
def informative_name_tokens(text):

    if pd.isna(text) or not str(text).strip():
        return []

    text = normalize_name(text)

    return [
        token
        for token in set(text.split())
        if len(token) >= 3
        and token not in NAME_STOPWORDS
    ]

name_token_index = defaultdict(list)

for row in candidate_source.reset_index().itertuples(index=False):

    tokens = informative_name_tokens(
        row.business_name
    )

    for token in tokens:
        name_token_index[
            (row.country, token)
        ].append(row.entity_id)

NAME_TOKEN_THRESHOLD = 500

name_token_index = {
    key: ids
    for key, ids in name_token_index.items()
    if len(ids) <= NAME_TOKEN_THRESHOLD
}

print("Selective name-token keys:", len(name_token_index))

In [ ]:
ADDRESS_STOPWORDS = {
    "road","street","avenue","boulevard","drive","lane","highway",
    "parkway","court","place","trail","circle","suite","apartment",
    "floor","flat","unit","block","plot","sector","nagar","colony",
    "near","opposite","opp","main","city","county","north","south",
    "east","west","the","and","for","null"
}
def informative_address_tokens(text):

    if pd.isna(text) or not str(text).strip():
        return []

    text = normalize_address(text)

    return [
        token
        for token in set(text.split())
        if len(token) >= 3
        and token not in ADDRESS_STOPWORDS
    ]

pair_counter = Counter()

for row in candidate_source.reset_index().itertuples(index=False):

    tokens = informative_address_tokens(
        row.business_address
    )

    if len(tokens) < 2:
        continue

    for t1, t2 in combinations(sorted(tokens), 2):

        pair_counter[
            (row.country, t1, t2)
        ] += 1

PAIR_THRESHOLD = 50

address_pair_index = defaultdict(list)

for row in candidate_source.reset_index().itertuples(index=False):

    tokens = informative_address_tokens(
        row.business_address
    )

    if len(tokens) < 2:
        continue

    for t1, t2 in combinations(sorted(tokens), 2):

        key = (row.country, t1, t2)

        if pair_counter[key] <= PAIR_THRESHOLD:
            address_pair_index[key].append(
                row.entity_id
            )

print(
    "Address pair keys:",
    len(address_pair_index)
)

In [ ]:
def get_candidates(row):

    country = row.country

    result = set()

    # A: exact normalized name
    name = normalize_name(row.business_name)

    if name:
        result.update(
            name_index.get(
                (country, name),
                []
            )
        )

    # B: exact normalized address
    address = normalize_address(row.business_address)

    if address:
        result.update(
            address_index.get(
                (country, address),
                []
            )
        )

    # C: informative name token
    for token in informative_name_tokens(
        row.business_name
    ):

        result.update(
            name_token_index.get(
                (country, token),
                []
            )
        )

    # D: address token pair
    tokens = informative_address_tokens(
        row.business_address
    )

    if len(tokens) >= 2:

        for t1, t2 in combinations(
            sorted(tokens), 2
        ):

            result.update(
                address_pair_index.get(
                    (country, t1, t2),
                    []
                )
            )

    return result

In [ ]:
rng = np.random.default_rng(42)

train_sample_ids = rng.choice(
    list(train_s1_ids),
    size=10_000,
    replace=False
)

train_sample_ids = set(train_sample_ids)

print(len(train_sample_ids))

s1_sample = s1[
    s1["entity_id"].isin(train_sample_ids)
].copy()

print(s1_sample.shape)

In [ ]:
candidate_rows = []

for row in s1_sample.itertuples(index=False):

    true_matches = ground_truth.get(
        row.entity_id,
        set()
    )

    candidates_for_s1 = get_candidates(row)

    for candidate_id in candidates_for_s1:

        candidate_rows.append(
            (
                row.entity_id,
                candidate_id,
                int(candidate_id in true_matches)
            )
        )

print(
    "Candidate pairs:",
    len(candidate_rows)
)

In [ ]:
training_pairs = pd.DataFrame(
    candidate_rows,
    columns=[
        "source1_entity_id",
        "candidate_entity_id",
        "label"
    ]
)

print(training_pairs.shape)
print(training_pairs["label"].value_counts())

In [ ]:
print(training_pairs["label"].value_counts())
print()
print(training_pairs["label"].value_counts(normalize=True) * 100)

In [ ]:
from functools import lru_cache
import re
!pip install rapidfuzz
from rapidfuzz.fuzz import ratio, token_set_ratio, token_sort_ratio

In [ ]:
s1_lookup = (
    s1.set_index("entity_id")[
        ["business_name", "business_address", "country"]
    ]
    .to_dict("index")
)

candidate_lookup = (
    candidate_source[
        ["business_name", "business_address", "country"]
    ]
    .to_dict("index")
)

In [ ]:
def safe_text(value):
    if pd.isna(value):
        return ""
    return str(value).strip()

def token_jaccard(a, b):
    a = safe_text(a)
    b = safe_text(b)

    if not a or not b:
        return 0.0

    a_tokens = set(a.split())
    b_tokens = set(b.split())

    union = a_tokens | b_tokens

    if not union:
        return 0.0

    return len(a_tokens & b_tokens) / len(union)

def shared_token_count(a, b):
    a = safe_text(a)
    b = safe_text(b)

    if not a or not b:
        return 0

    return len(
        set(a.split()) & set(b.split())
    )

def length_ratio(a, b):

    a = safe_text(a)
    b = safe_text(b)

    if not a or not b:
        return 0.0

    return min(len(a), len(b)) / max(
        len(a), len(b)
    )

In [ ]:
positive_pair = training_pairs[
    training_pairs["label"] == 1
].iloc[0]

negative_pair = training_pairs[
    training_pairs["label"] == 0
].iloc[0]

print("POSITIVE")
print(
    make_features(
        s1_lookup[
            positive_pair.source1_entity_id
        ],
        candidate_lookup[
            positive_pair.candidate_entity_id
        ]
    )
)

print("\nNEGATIVE")
print(
    make_features(
        s1_lookup[
            negative_pair.source1_entity_id
        ],
        candidate_lookup[
            negative_pair.candidate_entity_id
        ]
    )
)

In [ ]:
print(training_pairs.shape)
print(training_pairs.columns.tolist())